In [1]:
# Cell 1 — verify all libraries installed correctly
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
import shap

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("seaborn:", sns.__version__)
print("All libraries loaded successfully ✓")


pandas: 3.0.1
numpy: 2.4.3
seaborn: 0.13.2
All libraries loaded successfully ✓


In [2]:
# Cell 2 — load all 9 CSV files
data_path = "../data/"

customers = pd.read_csv(data_path + "olist_customers_dataset.csv")
orders = pd.read_csv(data_path + "olist_orders_dataset.csv")
order_items = pd.read_csv(data_path + "olist_order_items_dataset.csv")
payments = pd.read_csv(data_path + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(data_path + "olist_order_reviews_dataset.csv")
products = pd.read_csv(data_path + "olist_products_dataset.csv")
sellers = pd.read_csv(data_path + "olist_sellers_dataset.csv")
geolocation = pd.read_csv(data_path + "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(data_path + "product_category_name_translation.csv")

print("customers:", customers.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("payments:", payments.shape)
print("reviews:", reviews.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)
print("category_translation:", category_translation.shape)
print("\nAll 9 files loaded ✓")

customers: (99441, 5)
orders: (99441, 8)
order_items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)

All 9 files loaded ✓


In [3]:
# Cell 3 — understand the orders table
print("=== ORDERS TABLE ===")
print(orders.head(3))
print("\nColumns:", orders.columns.tolist())
print("\nData types:\n", orders.dtypes)
print("\nNull counts:\n", orders.isnull().sum())

=== ORDERS TABLE ===
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08-07 15:27:45   
2          2018-08-08 13:50:00           2018-08-17 18:06:29   

  order_estimated_delivery_date  
0           2017-10-18 00:00:00  
1           2018-08-13 00:00:00  
2           2018-09-04 00:00:00  

Columns: ['order_id', 'customer_id'

In [4]:
# Cell 4 — understand customers and order items
print("=== CUSTOMERS TABLE ===")
print(customers.head(3))
print("\n=== ORDER ITEMS TABLE ===")
print(order_items.head(3))
print("\n=== PAYMENTS TABLE ===")
print(payments.head(3))

=== CUSTOMERS TABLE ===
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  

=== ORDER ITEMS TABLE ===
                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   

                         product_id                         seller_id  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
1  e5f2d52b802189ee65

In [5]:
# Cell 5 — merge all tables into master dataframe
# Step 1: orders + customers
df = orders.merge(customers, on="customer_id", how="left")

# Step 2: + order items
df = df.merge(order_items, on="order_id", how="left")

# Step 3: + payments
payments_agg = payments.groupby("order_id").agg(
    total_payment=("payment_value", "sum"),
    payment_installments=("payment_installments", "max"),
    payment_type=("payment_type", "first")
).reset_index()
df = df.merge(payments_agg, on="order_id", how="left")

# Step 4: + reviews
reviews_agg = reviews.groupby("order_id").agg(
    review_score=("review_score", "mean")
).reset_index()
df = df.merge(reviews_agg, on="order_id", how="left")

# Step 5: + products + category translation
products_merged = products.merge(category_translation, on="product_category_name", how="left")
df = df.merge(products_merged[["product_id", "product_category_name_english"]], on="product_id", how="left")

print("Master dataframe shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

Master dataframe shape: (113425, 23)

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'total_payment', 'payment_installments', 'payment_type', 'review_score', 'product_category_name_english']

First 3 rows:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2

In [6]:
# Cell 6 — save master dataframe to data folder
df.to_csv("../data/master_df.csv", index=False)
print("Master dataframe saved ✓")
print("Shape:", df.shape)
print("\nNull counts:\n", df.isnull().sum())

Master dataframe saved ✓
Shape: (113425, 23)

Null counts:
 order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 161
order_delivered_carrier_date     1968
order_delivered_customer_date    3229
order_estimated_delivery_date       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
order_item_id                     775
product_id                        775
seller_id                         775
shipping_limit_date               775
price                             775
freight_value                     775
total_payment                       3
payment_installments                3
payment_type                        3
review_score                      961
product_category_name_english    2402
dtype: int64


In [7]:
# Cell 7 — fix date columns and data types
# Convert date columns from string to datetime
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])
df["order_delivered_customer_date"] = pd.to_datetime(df["order_delivered_customer_date"])
df["order_estimated_delivery_date"] = pd.to_datetime(df["order_estimated_delivery_date"])

print("Date columns converted ✓")
print(df[["order_purchase_timestamp", "order_delivered_customer_date"]].dtypes)

Date columns converted ✓
order_purchase_timestamp         datetime64[us]
order_delivered_customer_date    datetime64[us]
dtype: object


In [8]:
# Cell 8 — define churn
# Business decision: customer who made no purchase in last 6 months = churned
# Reference date = most recent order date in the dataset
reference_date = df["order_purchase_timestamp"].max()
print("Reference date (most recent order):", reference_date)

# Get each customer's last purchase date
last_purchase = df.groupby("customer_unique_id")["order_purchase_timestamp"].max().reset_index()
last_purchase.columns = ["customer_unique_id", "last_purchase_date"]

# Calculate days since last purchase
last_purchase["days_since_purchase"] = (reference_date - last_purchase["last_purchase_date"]).dt.days

# Define churn: no purchase in last 180 days (6 months)
last_purchase["churned"] = (last_purchase["days_since_purchase"] > 180).astype(int)

print("\nChurn distribution:")
print(last_purchase["churned"].value_counts())
print("\nChurn rate:", round(last_purchase["churned"].mean() * 100, 2), "%")

Reference date (most recent order): 2018-10-17 17:30:18

Churn distribution:
churned
1    68115
0    27981
Name: count, dtype: int64

Churn rate: 70.88 %


In [9]:
# Cell 9 — handle missing values
print("Nulls before cleaning:\n", df.isnull().sum()[df.isnull().sum() > 0])

# Fill missing review scores with median
df["review_score"] = df["review_score"].fillna(df["review_score"].median())

# Fill missing category with unknown
df["product_category_name_english"] = df["product_category_name_english"].fillna("unknown")

# Drop rows where payment is missing
df = df.dropna(subset=["total_payment"])

print("\nNulls after cleaning:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nShape after cleaning:", df.shape)

Nulls before cleaning:
 order_approved_at                 161
order_delivered_carrier_date     1968
order_delivered_customer_date    3229
order_item_id                     775
product_id                        775
seller_id                         775
shipping_limit_date               775
price                             775
freight_value                     775
total_payment                       3
payment_installments                3
payment_type                        3
review_score                      961
product_category_name_english    2402
dtype: int64

Nulls after cleaning:
 order_approved_at                 161
order_delivered_carrier_date     1968
order_delivered_customer_date    3229
order_item_id                     775
product_id                        775
seller_id                         775
shipping_limit_date               775
price                             775
freight_value                     775
dtype: int64

Shape after cleaning: (113422, 23)


In [10]:
# Cell 10 — merge churn label into master dataframe and save
df = df.merge(last_purchase[["customer_unique_id", "days_since_purchase", "churned"]], 
              on="customer_unique_id", how="left")

print("Final dataframe shape:", df.shape)
print("\nChurn column added:")
print(df["churned"].value_counts())

# Save cleaned master dataframe
df.to_csv("../data/master_df_clean.csv", index=False)
print("\nClean master dataframe saved ✓")

Final dataframe shape: (113422, 25)

Churn column added:
churned
1    80157
0    33265
Name: count, dtype: int64

Clean master dataframe saved ✓
